# Notebook 2 — ETL: Preprocessing & PostgreSQL Storage
Extracts from MongoDB, transforms, loads into PostgreSQL.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.etl_pipeline import ETLPipeline

In [ ]:
etl = ETLPipeline()
etl.extract()
print('WHO shape:',     etl.who_df.shape)
print('WorldBank shape:',etl.wb_df.shape)
print('Disease shape:', etl.disease_df.shape)

## Missing Data Analysis

In [ ]:
import matplotlib.pyplot as plt
who_clean = etl._clean_who()
missing_pct = who_clean.isnull().mean().sort_values(ascending=False) * 100
print('Missing data (%):')
print(missing_pct[missing_pct > 0])

## Run Full Transformation and Load

In [ ]:
etl.transform()
print('Country profiles shape:', etl.country_profiles.shape)
etl.country_profiles.describe()

In [ ]:
etl.load()
print('ETL complete — data loaded into PostgreSQL')

## Verify PostgreSQL

In [ ]:
from sqlalchemy import create_engine, text
from config import PG_CONN_STR
engine = create_engine(PG_CONN_STR)
with engine.connect() as conn:
    result = conn.execute(text('SELECT table_name FROM information_schema.tables WHERE table_schema = \'public\''))
    tables = [r[0] for r in result]
print('PostgreSQL tables:', tables)
pd.read_sql('SELECT COUNT(*) as n FROM analysis_country_profiles', engine)